# 07 — Pruebas de funcionamiento y evidencia

Esta ejecución produce la evidencia de nuestra solución: 14 escenarios de texto y seis escenarios de voz con estado inicial reproducible; la comparación de voz con su transcripción correcta; nDCG@10, restricciones, errores ASR, calidad de voz y latencia por etapa. Los resultados se generan al ejecutar este notebook y no se reemplazan por cifras históricas.


In [1]:
%pip -q install -r requirements-lab4-modular.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements-lab4-modular.txt'
Note: you may need to restart the kernel to use updated packages.


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src" / "lab4_asistente").is_dir():
    ROOT = next((p for p in ROOT.parents if (p / "src" / "lab4_asistente").is_dir()), ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Raíz del proyecto:", ROOT)
from pathlib import Path
import importlib.metadata
import pandas as pd
from IPython.display import display
from lab4_asistente import config as C
from lab4_asistente.escenarios import ESCENARIOS, ejecutar
from lab4_asistente.evaluacion import (tabla_escenarios, resumen_escenarios,
    tabla_asr, tabla_latencias, tabla_calidad_tts, tabla_aciertos_fallos)
from lab4_asistente.runtime import cargar_sistema, cargar_modelos_voz
from lab4_asistente.voz import Semejanza


Raíz del proyecto: /home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz


## Seleccionar el perfil de voz

El menú se alimenta de las grabaciones disponibles. La siguiente celda carga los modelos y comprueba los audios de los seis escenarios.


In [3]:
import ipywidgets as widgets
from IPython.display import display

VOICES = C.voces_disponibles()
if not VOICES:
    raise FileNotFoundError("No se encontraron perfiles/grabaciones de voz.")
voice_selector = widgets.Dropdown(options=sorted(VOICES), description="Perfil de voz:")
display(voice_selector)
print("Elige la voz y ejecuta la siguiente celda para cargar modelos y escenarios.")


Dropdown(description='Perfil de voz:', options=('ana', 'diego', 'elsa', 'estuardo', 'nicole', 'winz'), value='…

Elige la voz y ejecuta la siguiente celda para cargar modelos y escenarios.


## Cargar modelos y preparar escenarios de voz

Qwen usa la GPU. Recuperación, Whisper y XTTS usan CPU durante esta evaluación para evitar que la carga simultánea exceda la VRAM local. Se miden las cargas iniciales por separado de la latencia de cada turno.


In [2]:
COMP = cargar_sistema(cargar_llm=True)
SIS = COMP["sistema"]

GPU lista: NVIDIA GeForce RTX 5070 Ti Laptop GPU | VRAM libre: 10.8/11.9 GB


/home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/.venv-dl-cuda/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [4]:
COMP = cargar_sistema(cargar_llm=True)
SIS = COMP["sistema"]
VOICE_ID = voice_selector.value
VOZ = C.rutas_voz(VOICE_ID)
ASR, TTS, cargas_voz = cargar_modelos_voz(VOICE_ID)
CARGAS_INICIALES = {**COMP["cargas"], **cargas_voz}
RUTA_AUDIOS = VOZ["audios"]
AUDIO_REQUERIDO = [(e["id"], t["audio"]) for e in ESCENARIOS if e["modo"] == "voz" for t in e["turnos"]]
FALTAN = [nombre for _, nombre in AUDIO_REQUERIDO if not (RUTA_AUDIOS / nombre).exists()]
print("Voz seleccionada:", VOICE_ID)
print("Dispositivos:", COMP["dispositivos"], "ASR/TTS:", cargas_voz["dispositivo_asr"], cargas_voz["dispositivo_tts"])
print("Cargas iniciales (por separado de latencias):", CARGAS_INICIALES)
print("Audios requeridos:", len(AUDIO_REQUERIDO), "| faltantes:", len(FALTAN))
if FALTAN:
    display(pd.DataFrame({"audios_faltantes": FALTAN}))


GPU lista: NVIDIA GeForce RTX 5070 Ti Laptop GPU | VRAM libre: 10.8/11.9 GB


/home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/.venv-dl-cuda/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

 > tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.
 > Using model: xtts


/home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/.venv-dl-cuda/lib/python3.11/site-packages/TTS/tts/layers/xtts/xtts_manager.py:5: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  self.speakers = torch.load(speaker_file_path)
/home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/.venv-dl-cuda/lib/python3.11/site-packages/TTS/utils/io.py:54: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  return torch.load(f, map_location=map_location, **kwargs)
GPT2InferenceModel has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, a

Voz seleccionada: ana
Dispositivos: {'llm': 'cuda:0', 'recuperacion': 'cpu'} ASR/TTS: cpu cpu
Cargas iniciales (por separado de latencias): {'recuperacion': 5.683711744999528, 'llm': 12.05599724700005, 'asr': 3.1157072200003313, 'tts': 20.011231976999625, 'dispositivo_asr': 'cpu', 'dispositivo_tts': 'cpu'}
Audios requeridos: 23 | faltantes: 23


,audios_faltantes
0,E01_t1.wav
1,E01_t2.wav
2,E01_t3.wav
3,E02_t1.wav
4,E02_t2.wav
5,E02_t3.wav
6,E02_t4.wav
7,E03_t1.wav
8,E03_t2.wav
9,E03_t3.wav


## Ejecutar los 14 escenarios de texto

In [5]:
C.RUTA_ESCENARIOS.mkdir(parents=True, exist_ok=True)
RES_TEXTO = []
for e in [x for x in ESCENARIOS if x["modo"] == "texto"]:
    RES_TEXTO.append(ejecutar(e, SIS, "texto", C.RUTA_ESCENARIOS / "texto" / e["id"]))
print("Escenarios de texto ejecutados:", len(RES_TEXTO))


Escenarios de texto ejecutados: 14


## Ejecutar los seis escenarios con audio

Las grabaciones se conservan junto con transcripciones y operaciones. Se ejecutan también con sus textos correctos para separar errores del ASR de errores del agente.

In [6]:
print("Carpeta de audios:", RUTA_AUDIOS)
print("Grabaciones pendientes:")
for archivo in FALTAN:
    print(archivo)

Carpeta de audios: /home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/data/scenario_audio/ana
Grabaciones pendientes:
E01_t1.wav
E01_t2.wav
E01_t3.wav
E02_t1.wav
E02_t2.wav
E02_t3.wav
E02_t4.wav
E03_t1.wav
E03_t2.wav
E03_t3.wav
E03_t4.wav
E04_t1.wav
E04_t2.wav
E04_t3.wav
E05_t1.wav
E05_t2.wav
E05_t3.wav
E06_t1.wav
E06_t2.wav
E06_t3.wav
E06_t4.wav
E06_t5.wav
E06_t6.wav


In [7]:
from pprint import pprint

for escenario in ESCENARIOS:
    if escenario["modo"] == "voz":
        pprint(escenario, sort_dicts=False)
        print()

{'id': 'E01',
 'modo': 'voz',
 'tipo': 'compra que cumple restricciones',
 'presupuesto': 30,
 'turnos': [{'texto': 'Quiero comprar dos paquetes de pilas AA, mi presupuesto '
                      'es de treinta euros.',
             'audio': 'E01_t1.wav'},
            {'texto': 'Agrega la primera opción y prepárame la cotización.',
             'audio': 'E01_t2.wav'},
            {'texto': 'Sí, confirmo el pedido.', 'audio': 'E01_t3.wav'}],
 'esperado': {'pedidos': 1,
              'items': [{'palabras': ['pilas'], 'cantidad': 2}],
              'total_max': 30}}

{'id': 'E02',
 'modo': 'voz',
 'tipo': 'solicitud ambigua con aclaración',
 'turnos': [{'texto': 'Quiero comprar una funda.', 'audio': 'E02_t1.wav'},
            {'texto': 'Es para un iPhone doce, de color negro.',
             'audio': 'E02_t2.wav'},
            {'texto': 'Agrega la primera y dime el total.',
             'audio': 'E02_t3.wav'},
            {'texto': 'Sí, confirmo.', 'audio': 'E02_t4.wav'}],
 'esperado': {'

In [8]:
esperados = [
    turno["audio"]
    for escenario in ESCENARIOS
    if escenario["modo"] == "voz"
    for turno in escenario["turnos"]
]

for persona in ["diego", "elsa", "nicole"]:
    carpeta = ROOT / "data" / "scenario_audio" / persona
    faltantes = [
        nombre for nombre in esperados
        if not (carpeta / nombre).is_file()
    ]
    print(f"{persona}: {len(esperados) - len(faltantes)}/{len(esperados)}")
    if faltantes:
        print("  Faltan:", faltantes)

diego: 23/23
elsa: 23/23
nicole: 23/23


In [10]:
# Seleccionar al comprador que vamos a evaluar.
PERSONA = "nicole"
RUTA_AUDIOS = ROOT / "data" / "scenario_audio" / PERSONA

# Actualizar la variable que utiliza la evaluación.
FALTAN = [
    nombre for nombre in esperados
    if not (RUTA_AUDIOS / nombre).is_file()
]

print("Persona:", PERSONA)
print("Ruta:", RUTA_AUDIOS)
print("FALTAN:", FALTAN)

Persona: nicole
Ruta: /home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/data/scenario_audio/nicole
FALTAN: []


In [12]:
import time
import pandas as pd

# Verificar nuevamente los archivos de la persona seleccionada.
escenarios_voz = [e for e in ESCENARIOS if e["modo"] == "voz"]

FALTAN = [
    turno["audio"]
    for e in escenarios_voz
    for turno in e["turnos"]
    if not (RUTA_AUDIOS / turno["audio"]).is_file()
]
assert not FALTAN, f"Faltan audios en {RUTA_AUDIOS}: {FALTAN}"

RES_VOZ, RES_VOZ_TEXTO = [], []

print(f"Comprador: {PERSONA}", flush=True)
print(f"Audios: {RUTA_AUDIOS}", flush=True)

for i, e in enumerate(escenarios_voz, start=1):
    inicio = time.perf_counter()

    print(
        f"\n[{i}/{len(escenarios_voz)}] {e['id']}: {e['tipo']}",
        flush=True,
    )
    print("Procesando audio y generando respuestas...", flush=True)

    carpeta = C.RUTA_ESCENARIOS / "voz_audio" / PERSONA / e["id"]
    resultado_audio = ejecutar(
        e, SIS, "voz", carpeta,
        RUTA_AUDIOS, ASR, TTS,
        sintetizar=True,
    )
    RES_VOZ.append(resultado_audio)

    print("Repitiendo el escenario con el texto correcto...", flush=True)

    carpeta_txt = (
        C.RUTA_ESCENARIOS
        / "voz_transcripcion_correcta"
        / PERSONA
        / e["id"]
    )
    resultado_texto = ejecutar(e, SIS, "texto", carpeta_txt)
    RES_VOZ_TEXTO.append(resultado_texto)

    print(
        f"Finalizado en {time.perf_counter() - inicio:.1f} segundos"
        f" | Audio: {resultado_audio['completado']}"
        f" | Texto: {resultado_texto['completado']}",
        flush=True,
    )

COMPARACION_ASR = pd.DataFrame([
    {
        "persona": PERSONA,
        "escenario": audio["id"],
        "completado_con_audio": audio["completado"],
        "completado_con_transcripcion_correcta": texto["completado"],
        "resultado_distinto": audio["completado"] != texto["completado"],
    }
    for audio, texto in zip(RES_VOZ, RES_VOZ_TEXTO)
])

ruta_comparacion = (
    C.RUTA_ESCENARIOS / "voz_audio" / PERSONA / "comparacion_asr.csv"
)
ruta_comparacion.parent.mkdir(parents=True, exist_ok=True)
COMPARACION_ASR.to_csv(ruta_comparacion, index=False)

display(COMPARACION_ASR)
print("Comparación guardada en:", ruta_comparacion)

Comprador: nicole
Audios: /home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/data/scenario_audio/nicole

[1/6] E01: compra que cumple restricciones
Procesando audio y generando respuestas...


/home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/.venv-dl-cuda/lib/python3.11/site-packages/transformers/models/whisper/generation_whisper.py:509: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
You have passed task=transcribe, but also have set `forced_decoder_ids` to [[1, None], [2, 50360]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Repitiendo el escenario con el texto correcto...
Finalizado en 261.9 segundos | Audio: False | Texto: True

[2/6] E02: solicitud ambigua con aclaración
Procesando audio y generando respuestas...


/home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/.venv-dl-cuda/lib/python3.11/site-packages/transformers/models/whisper/generation_whisper.py:509: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(


Repitiendo el escenario con el texto correcto...
Finalizado en 177.7 segundos | Audio: True | Texto: True

[3/6] E03: cambio de cantidad conservando restricciones
Procesando audio y generando respuestas...


/home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/.venv-dl-cuda/lib/python3.11/site-packages/transformers/models/whisper/generation_whisper.py:509: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(


... (5 outputs left out)

/home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/.venv-dl-cuda/lib/python3.11/site-packages/transformers/models/whisper/generation_whisper.py:509: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(


Repitiendo el escenario con el texto correcto...
Finalizado en 362.8 segundos | Audio: False | Texto: False


,persona,escenario,completado_con_audio,completado_con_transcripcion_correcta,resultado_distinto
0,nicole,E01,False,True,True
1,nicole,E02,True,True,False
2,nicole,E03,False,False,False
3,nicole,E04,False,False,False
4,nicole,E05,False,False,False
5,nicole,E06,False,False,False


Comparación guardada en: /home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/output/escenarios/voz_audio/nicole/comparacion_asr.csv


In [14]:
from pprint import pprint

# E01: comparar qué ocurrió con audio y con texto.
for etiqueta, resultados in [
    ("AUDIO", RES_VOZ),
    ("TEXTO CORRECTO", RES_VOZ_TEXTO),
]:
    print(f"\n===== E01 — {etiqueta} =====")
    pprint(
        next(r for r in resultados if r["id"] == "E01"),
        sort_dicts=False,
    )

# E03: revisar primero el fallo que también aparece con texto.
print("\n===== E03 — TEXTO CORRECTO =====")
pprint(
    next(r for r in RES_VOZ_TEXTO if r["id"] == "E03"),
    sort_dicts=False,
)


===== E01 — AUDIO =====
{'id': 'E01',
 'modo': 'voz',
 'tipo': 'compra que cumple restricciones',
 'turnos': 3,
 'completado': False,
 'fallos': "pedidos 0 (esperado 1); sin item ['pilas']",
 'incumplimientos': '',
 'pedidos': 0,
 'registro': [{'turno': 1,
               'texto_correcto': 'Quiero comprar dos paquetes de pilas AA, mi '
                                 'presupuesto es de treinta euros.',
               'eventos': [],
               'transcripcion_asr': 'Quiero comprar dos paquetes de pilas AA. '
                                    'Mi presupuesto es de 30 euros.',
               'entrada': 'Quiero comprar dos paquetes de pilas AA. Mi '
                          'presupuesto es de 30 euros.',
               'respuesta': 'Encontré varias opciones. ¿Te interesa el combo '
                            'de POWEROWL por 21,99 euros o las pilas Amazon '
                            'Basics recargables por 7,99 euros?',
               'aclaracion': True,
               'pasos_llm

## Resumen, incumplimientos y métricas de ASR

In [15]:
TODOS = RES_TEXTO + RES_VOZ
MATRIZ = tabla_escenarios(TODOS)
display(MATRIZ)
print("Resumen de escenarios:", resumen_escenarios(TODOS))

ASR_METRICAS = tabla_asr(RES_VOZ)
display(ASR_METRICAS)
if len(ASR_METRICAS):
    print("WER medio:", ASR_METRICAS.WER.mean())
    print("CER medio:", ASR_METRICAS.CER.mean())
    print("Errores de cantidades/importes/productos:",
          ASR_METRICAS.err_cantidades.sum(), ASR_METRICAS.err_importes.sum(), ASR_METRICAS.err_productos.sum())

print("Comparación de audio frente al texto correcto:")
display(COMPARACION_ASR)


,id,modo,tipo,completado,fallos,incumplimientos,pedidos
0,T07,texto,compra simple con presupuesto,True,,,1
1,T08,texto,varios productos con presupuesto total,True,,,1
2,T09,texto,presupuesto insuficiente,False,turno 1: la respuesta no menciona ['presupuest...,,0
3,T10,texto,atributo no verificable,False,"turno 1: la respuesta no menciona ['descono', ...",,0
4,T11,texto,agotado sin solución,False,"turno 1: la respuesta no menciona ['agotad', '...",,0
5,T12,texto,cantidad mayor que el stock,False,pedidos 0 (esperado 1); sin item ['regleta'],,0
6,T13,texto,compra sin cotización previa (consentimiento),False,pedidos 0 (esperado 1); sin item ['pilas'],,0
7,T14,texto,quitar un producto del carrito,False,pedidos 0 (esperado 1); sin item ['altavoz']; ...,,0
8,T15,texto,cambio de preferencia (color),False,"sin item ['mochila', 'negro']",,1
9,T16,texto,comparar alternativas,True,,,0


Resumen de escenarios: {'total': 20, 'completados': 7, 'porcentaje_completado': 35.0, 'incumplimientos': 0}


,escenario,turno,referencia,transcripcion,WER,CER,err_cantidades,err_importes,err_productos,n_cantidades,n_importes,n_productos
0,E01,1,"Quiero comprar dos paquetes de pilas AA, mi pr...",Quiero comprar dos paquetes de pilas AA. Mi pr...,0.076923,0.094595,0,0,0,1,1,2
1,E01,2,Agrega la primera opción y prepárame la cotiza...,Agrega la primera opción y prepárame la cotiza...,0.000000,0.000000,0,0,0,0,0,0
2,E01,3,"Sí, confirmo el pedido.","Sí, confirmo el pedido.",0.000000,0.000000,0,0,0,0,0,0
3,E02,1,Quiero comprar una funda.,Quiero comprar una funda.,0.000000,0.000000,0,0,0,1,0,1
4,E02,2,"Es para un iPhone doce, de color negro.",Es para un iPhone 12 de color negro.,0.125000,0.108108,0,0,0,2,0,3
5,E02,3,Agrega la primera y dime el total.,Agrega la primera y dime el total.,0.000000,0.000000,0,0,0,0,0,1
6,E02,4,"Sí, confirmo.","Sí, confirmo.",0.000000,0.000000,0,0,0,0,0,0
7,E03,1,"Necesito tres botellas de agua para niños, con...","Necesito tres botellas de agua para niños, con...",0.066667,0.097561,0,0,0,2,1,3
8,E03,2,Agrega la más barata.,Agrégala más barata.,0.500000,0.050000,0,0,0,0,0,0
9,E03,3,"Mejor que sean dos en lugar de tres, y prepára...",Mejor que sean 2 en lugar de 3 y prepárame la ...,0.166667,0.114754,0,0,0,2,0,1


WER medio: 0.058914636088549126
CER medio: 0.026203587830609807
Errores de cantidades/importes/productos: 0 0 1
Comparación de audio frente al texto correcto:


,escenario,completado_con_audio,completado_con_transcripcion_correcta,diferencia_por_asr
0,E01,False,True,True
1,E02,True,True,False
2,E03,False,False,False
3,E04,False,False,False
4,E05,False,False,False
5,E06,False,False,False


## Latencia por etapa y evaluación de voz sintetizada

In [16]:
LATENCIAS = tabla_latencias(RES_VOZ)
display(LATENCIAS.groupby("etapa").segundos.agg(["mean", "median", "max"]).sort_values("mean", ascending=False))
print("Carga inicial separada de las latencias:", CARGAS_INICIALES)
SEMEJANZA = Semejanza(C.MODELO_SEMEJANZA, "cpu")
REF_VOZ = VOZ["referencia"]
TTS_METRICAS = tabla_calidad_tts(
    RES_VOZ,
    C.RUTA_ESCENARIOS / "voz_audio" / PERSONA,
    ASR,
    SEMEJANZA,
    REF_VOZ,
)
display(TTS_METRICAS)


,mean,median,max
etapa,,,
total,56.101870,50.610956,120.725254
tts,44.819359,40.707623,96.022192
asr,5.607599,5.277666,13.227084
llm,3.106422,2.764959,6.643610
recuperacion,2.561268,0.000000,14.342744
herramientas,0.007222,0.002683,0.038280


Carga inicial separada de las latencias: {'recuperacion': 5.683711744999528, 'llm': 12.05599724700005, 'asr': 3.1157072200003313, 'tts': 20.011231976999625, 'dispositivo_asr': 'cpu', 'dispositivo_tts': 'cpu'}


preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/405M [00:00<?, ?B/s]

/home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/.venv-dl-cuda/lib/python3.11/site-packages/transformers/models/whisper/generation_whisper.py:509: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
/home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/.venv-dl-cuda/lib/python3.11/site-packages/torch/nn/functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


,escenario,turno,WER_inteligibilidad,semejanza_coseno
0,E01,1,0.142857,0.972064
1,E01,2,0.074074,0.975323
2,E01,3,0.037037,0.976125
3,E02,1,0.000000,0.956514
4,E02,2,0.238095,0.955950
5,E02,3,0.000000,0.963347
6,E02,4,0.200000,0.947923
7,E03,1,0.325000,0.973215
8,E03,2,0.086957,0.978600
9,E03,3,0.285714,0.968365


## Análisis de cinco aciertos y cinco fallos

In [19]:
ANALISIS = tabla_aciertos_fallos(TODOS, limite=5)
print("Aciertos y fallos seleccionados desde esta ejecución. La causa se basa en el verificador y el registro real de herramientas.")
display(ANALISIS[["tipo", "escenario", "modo", "diagnostico_observado"]])
print(
    "Casos seleccionados de esta ejecución. "
    "El diagnóstico muestra condiciones incumplidas; "
    "la causa se revisa en las transcripciones y operaciones."
)

Aciertos y fallos seleccionados desde esta ejecución. La causa se basa en el verificador y el registro real de herramientas.


,tipo,escenario,modo,diagnostico_observado
0,acierto,T07,texto,El estado final pasó las condiciones definidas...
1,acierto,T08,texto,El estado final pasó las condiciones definidas...
2,acierto,T16,texto,El estado final pasó las condiciones definidas...
3,acierto,T17,texto,El estado final pasó las condiciones definidas...
4,acierto,T18,texto,El estado final pasó las condiciones definidas...
5,fallo,T09,texto,El estado final no alcanzó el resultado espera...
6,fallo,T10,texto,Una validación/herramienta informó: product_id...
7,fallo,T11,texto,El estado final no alcanzó el resultado espera...
8,fallo,T12,texto,El estado final no alcanzó el resultado espera...
9,fallo,T13,texto,El estado final no alcanzó el resultado espera...


Casos seleccionados de esta ejecución. El diagnóstico muestra condiciones incumplidas; la causa se revisa en las transcripciones y operaciones.


In [17]:
COMPARACION_ASR = pd.DataFrame([
    {
        "persona": PERSONA,
        "escenario": audio["id"],
        "completado_con_audio": audio["completado"],
        "completado_con_transcripcion_correcta": texto["completado"],
        "resultado_distinto": audio["completado"] != texto["completado"],
    }
    for audio, texto in zip(RES_VOZ, RES_VOZ_TEXTO)
])

display(COMPARACION_ASR)

,persona,escenario,completado_con_audio,completado_con_transcripcion_correcta,resultado_distinto
0,nicole,E01,False,True,True
1,nicole,E02,True,True,False
2,nicole,E03,False,False,False
3,nicole,E04,False,False,False
4,nicole,E05,False,False,False
5,nicole,E06,False,False,False


## Guardar resultados y configuración ejecutada

In [20]:
from datetime import datetime

RUTA_RESULTADOS_EJECUCION = (
    C.RUTA_RESULTADOS
    / PERSONA
    / datetime.now().strftime("%Y%m%d_%H%M%S")
)

RUTA_RESULTADOS_EJECUCION.mkdir(parents=True, exist_ok=True)
MATRIZ.to_csv(RUTA_RESULTADOS_EJECUCION / "metricas_escenarios.csv", index=False)
ASR_METRICAS.to_csv(RUTA_RESULTADOS_EJECUCION / "metricas_asr.csv", index=False)
LATENCIAS.to_csv(RUTA_RESULTADOS_EJECUCION / "latencias_por_turno.csv", index=False)
TTS_METRICAS.to_csv(RUTA_RESULTADOS_EJECUCION / "metricas_tts_por_respuesta.csv", index=False)
paquetes = ["torch", "torchaudio", "transformers", "sentence-transformers", "TTS", "pandas", "pyarrow", "rank-bm25"]
versiones = {}
for nombre in paquetes:
    try:
        versiones[nombre] = importlib.metadata.version(nombre)
    except importlib.metadata.PackageNotFoundError:
        versiones[nombre] = "no instalado"
(RUTA_RESULTADOS_EJECUCION / "versiones.json").write_text(pd.Series(versiones).to_json(indent=2), encoding="utf-8")
print("Resultados guardados en:", RUTA_RESULTADOS_EJECUCION)

COMPARACION_ASR.to_csv(RUTA_RESULTADOS_EJECUCION / "escenarios_audio_vs_transcripcion_correcta.csv", index=False)
ANALISIS.to_csv(RUTA_RESULTADOS_EJECUCION / "analisis_cinco_aciertos_y_fallos.csv", index=False)


Resultados guardados en: /home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/output/resultados/nicole/20261125_190507


In [21]:
import json

evidencia = {
    "comprador": PERSONA,
    "voz_asistente": VOICE_ID,
    "ruta_audios": str(RUTA_AUDIOS),
    "referencia_voz_asistente": str(REF_VOZ),
    "cargas_iniciales": CARGAS_INICIALES,
    "resultados_texto": RES_TEXTO,
    "resultados_voz": RES_VOZ,
    "resultados_replay": RES_VOZ_TEXTO,
}

(RUTA_RESULTADOS_EJECUCION / "evidencia.json").write_text(
    json.dumps(evidencia, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)

162263

In [3]:
RES_DIAGNOSTICO = []

for escenario in ESCENARIOS:
    if escenario["modo"] != "voz":
        continue

    print(f"Probando {escenario['id']} por texto...", flush=True)

    resultado = ejecutar(
        escenario,
        SIS,
        "texto",
        C.RUTA_ESCENARIOS / "diagnostico_texto" / escenario["id"],
    )
    RES_DIAGNOSTICO.append(resultado)

display(tabla_escenarios(RES_DIAGNOSTICO))

Probando E01 por texto...
Probando E02 por texto...
Probando E03 por texto...
Probando E04 por texto...
Probando E05 por texto...
Probando E06 por texto...


,id,modo,tipo,completado,fallos,incumplimientos,pedidos
0,E01,texto,compra que cumple restricciones,True,,,1
1,E02,texto,solicitud ambigua con aclaración,False,pedidos 0 (esperado 1); sin item ['funda'],,0
2,E03,texto,cambio de cantidad conservando restricciones,False,pedidos 0 (esperado 1); sin item ['botella'],,0
3,E04,texto,producto agotado con alternativa,False,pedidos 0 (esperado 1); sin item ['termo'],,0
4,E05,texto,corrección de transcripción antes de comprar,True,,,1
5,E06,texto,cambio de stock entre cotización y confirmación,False,pedidos 0 (esperado 1); sin item ['mochila'],,0


In [4]:
from pprint import pprint

for escenario_id in ["E01", "E02"]:
    print(f"\n===== {escenario_id} =====")
    resultado = next(
        r for r in RES_DIAGNOSTICO
        if r["id"] == escenario_id
    )

    for turno in resultado["registro"]:
        print(f"\nTurno {turno['turno']}: {turno['entrada']}")
        print("Respuesta:", turno["respuesta"])
        print("Operaciones:")
        pprint(turno["operaciones"], sort_dicts=False)


===== E01 =====

Turno 1: Quiero comprar dos paquetes de pilas AA, mi presupuesto es de treinta euros.
Respuesta: {"name": "registrarrestricciones", "arguments": {"!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
Operaciones:
[]

Turno 2: Agrega la primera opción y prepárame la cotización.
Respuesta: Ya hemos agregado las pilas a tu carrito. El costo total es de $15.98. ¿Estás seguro de querer confirmar este pedido?
Operaciones:
[{'paso': 1,
  'herramienta': 'buscar_productos',
  'argumentos': {'cantidad': 2,
                 'consulta': 'paquetes pilas AA',
                 'precio_max': 30,
                 'k': 1},
  'resultado': {'ok': True,
                'resultado

In [5]:
from pprint import pprint

for escenario_id in ["E03", "E04", "E06"]:
    print(f"\n===== {escenario_id} =====")
    resultado = next(r for r in RES_DIAGNOSTICO if r["id"] == escenario_id)

    for turno in resultado["registro"]:
        print(f"\nTurno {turno['turno']}: {turno['entrada']}")
        print("Respuesta:", turno["respuesta"])
        print("Operaciones:")
        pprint(turno["operaciones"], sort_dicts=False)


===== E03 =====

Turno 1: Necesito tres botellas de agua para niños, con un máximo de cuarenta euros en total.
Respuesta: Encontré varias opciones para usted. ¿Le gustaría agregar la primera opción, que cuesta 4.99 euros por unidad, o prefiere algo más caro?
Operaciones:
[{'paso': 1,
  'herramienta': 'registrar_restricciones',
  'argumentos': {'presupuesto_max': 40},
  'resultado': {'ok': True, 'presupuesto_max': 40.0, 'preferencias': []}},
 {'paso': 1,
  'herramienta': 'buscar_productos',
  'argumentos': {'cantidad': 3, 'consulta': 'botellas de agua para niños'},
  'resultado': {'ok': True,
                'resultados': [{'posicion': 1,
                                'product_id': 'B087TY5YXD',
                                'titulo': 'DigHealth DIY Botellas de Agua con '
                                          'Pegatinas, 500 ML Aluminum Botella '
                                          'Personalizada Sin BPA para Niña',
                                'marca': 'DigHealth',
  